# 02 · KD con backbone descongelado

`ALFA` y `TEMPERATURA` no cambian respecto a `01`, para que la única variable entre ambos sea el descongelado.

In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "corpus.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Raíz del proyecto:", ROOT)

Raíz del proyecto: /Users/cristiansandoval/Universidad/Semillero/ButterflyModeling


In [2]:
import platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import tensorflow as tf

from scripts import corpus, evalStats
from scripts import modelRegistry as registro
from scripts.architectures import buildModel, descongelarCola, gradcamLayer, preprocessFn
from scripts.dataPipeline import buildAugmenter, loadSplit, preparar, prepararCrudo
from scripts.distillation import Destilador
from scripts.vizStyle import SERIE_1, SERIE_2, applyStyle, plotConfusionMatrix

estudianteInfo = registro.readJson(registro.OUTPUTS_DIR / "student.json")
if estudianteInfo is None:
    raise FileNotFoundError(
        "Falta outputs/student.json — corre antes 01_student/05_comparacion_estudiante.ipynb")

ARQUITECTURA = estudianteInfo["architecture"]
RUN_NAME = f"kd-respuesta-ft-{ARQUITECTURA.replace('_', '-')}"
NOTAS = "KD de respuesta con la cola del backbone del estudiante descongelada"
ROL = "destilado"
PROMOVER = False

FRACCION_DESCONGELADA = 0.30
TEMPERATURA = 4.0
ALFA = 0.7

TAMANO = (320, 320)
LOTE = 32
EPOCAS = 50
PACIENCIA = 8
TASA_APRENDIZAJE = 1e-4
SEMILLA = 42

tf.keras.utils.set_random_seed(SEMILLA)
corpus.materializar()

RUN_ID = registro.buildRunId(RUN_NAME)
RUN_DIR = registro.createRunDir(RUN_ID)
IMGS = RUN_DIR / "imgs"

print(f"run_id : {RUN_ID}")
print(f"TF     : {tf.__version__} · dispositivos: "
      f"{[d.device_type for d in tf.config.list_physical_devices()]}")

run_id : 20260926T204944Z_kd-respuesta-ft-mobilenet-v3-small
TF     : 2.19.1 · dispositivos: ['CPU']


## 1 · Docente

Sale de `outputs/teacher.json`, escrito por `02_teacher/06`.

In [3]:
docenteInfo = registro.readJson(registro.OUTPUTS_DIR / "teacher.json")
if docenteInfo is None:
    raise FileNotFoundError("Falta outputs/teacher.json — corre antes 02_teacher/06_comparacion_docente.ipynb")

ARQUITECTURA_DOCENTE = docenteInfo["architecture"]
docente = tf.keras.models.load_model(
    registro.runDir(docenteInfo["run_id"]) / "model.keras"
)
docente.trainable = False

pd.Series({
    "run_id": docenteInfo["run_id"],
    "arquitectura": ARQUITECTURA_DOCENTE,
    "test_accuracy": round(docenteInfo["test_accuracy"], 4),
    "test_macro_f1": round(docenteInfo["test_macro_f1"], 4),
    "parametros": docente.count_params(),
}).to_frame("docente")

,docente
run_id,20260913T005433Z_efficientnet-v2-s
arquitectura,efficientnet_v2_s
test_accuracy,0.9142
test_macro_f1,0.9135
parametros,21165231


## 2 · Datos

`train` sale crudo (0-255) y aumentado una sola vez; el `Destilador` preprocesa por dentro para cada modelo. `validate`/`test` van preprocesados para el estudiante, sin barajar.

In [4]:
preprocessDocente = preprocessFn(ARQUITECTURA_DOCENTE)
preprocessEstudiante = preprocessFn(ARQUITECTURA)
augmentador = buildAugmenter()

dsTrainCrudo, clases = loadSplit("dataset/train", TAMANO, LOTE)
dsValCrudo, _ = loadSplit("dataset/validate", TAMANO, LOTE, shuffle=False)
dsTestCrudo, _ = loadSplit("dataset/test", TAMANO, LOTE, shuffle=False)

dsTrain = prepararCrudo(dsTrainCrudo, augmentador)
dsValDestilacion = dsValCrudo.prefetch(tf.data.AUTOTUNE)   # crudo: el Destilador preprocesa por dentro
dsVal = preparar(dsValCrudo, preprocessEstudiante)
dsTest = preparar(dsTestCrudo, preprocessEstudiante)

print(f"{len(clases)} clases")

Found 7218 files belonging to 79 classes.
Found 1038 files belonging to 79 classes.
Found 2062 files belonging to 79 classes.
79 clases


## 3 · Estudiante

In [5]:
estudiante = buildModel(ARQUITECTURA, len(clases))
infoFt = descongelarCola(estudiante, FRACCION_DESCONGELADA)
print(infoFt)

pd.Series({
    "estudiante": ARQUITECTURA,
    "parametros_estudiante": estudiante.count_params(),
    "parametros_docente": docente.count_params(),
    "razon_compresion": round(docente.count_params() / estudiante.count_params(), 1),
}).to_frame("valor")

/opt/miniconda3/envs/semillero/lib/python3.12/site-packages/keras/src/applications/mobilenet_v3.py:454: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


{'capas_backbone': 157, 'capas_descongeladas': 38, 'parametros_entrenables': 1200631}


,valor
estudiante,mobilenet_v3_small
parametros_estudiante,1412543
parametros_docente,21165231
razon_compresion,15.0


## 4 · Pesos de clase

In [6]:
pesosClase = corpus.classWeights()
pd.Series(
    {clases[i]: round(p, 3) for i, p in pesosClase.items()}
).sort_values(ascending=False).head(8).to_frame("peso")

,peso
autochton_itylus,3.151
taygetis_sylvia,3.151
cissia_confusa,3.151
pseudodebis_puritana,3.046
pyrgus_adepta,2.947
pteronymia_latilla,2.125
euselasia_mys,2.077
pteronymia_aletta,1.986


## 5 · Entrenamiento con destilación

`val_loss` es la CE del estudiante, no la pérdida de destilación: así el EarlyStopping monitorea lo mismo que en los notebooks de candidatos. El docente corre en inferencia en cada paso — no se puede precomputar, la vista aumentada cambia cada época.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

destilador = Destilador(
    docente, estudiante, preprocessDocente, preprocessEstudiante, TEMPERATURA, ALFA
)
destilador.compile(optimizer=Adam(learning_rate=TASA_APRENDIZAJE))

callbacks = [
    EarlyStopping(monitor="val_loss", patience=PACIENCIA,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3,
                      min_lr=1e-6, verbose=1),
]

inicio = datetime.now(timezone.utc)
ajuste = destilador.fit(
    dsTrain,
    validation_data=dsValDestilacion,
    epochs=EPOCAS,
    callbacks=callbacks,
    class_weight=pesosClase,
)
fin = datetime.now(timezone.utc)

historia = {k: [float(v) for v in vs] for k, vs in ajuste.history.items()}
print(f"\n{len(historia['loss'])} épocas en {(fin - inicio).total_seconds() / 60:.1f} min")

Epoch 1/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 1236s 5s/step - accuracy: 0.0234 - loss: 7.2559 - val_accuracy: 0.1050 - val_loss: 3.8734 - learning_rate: 1.0000e-04
Epoch 2/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 1196s 5s/step - accuracy: 0.0629 - loss: 6.5530 - val_accuracy: 0.2341 - val_loss: 3.1574 - learning_rate: 1.0000e-04
Epoch 3/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 1147s 5s/step - accuracy: 0.1185 - loss: 5.9722 - val_accuracy: 0.3439 - val_loss: 2.7215 - learning_rate: 1.0000e-04
Epoch 4/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 1114s 5s/step - accuracy: 0.1845 - loss: 5.5377 - val_accuracy: 0.4258 - val_loss: 2.4113 - learning_rate: 1.0000e-04
Epoch 5/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 1102s 5s/step - accuracy: 0.2354 - loss: 5.1740 - val_accuracy: 0.4788 - val_loss: 2.1539 - learning_rate: 1.0000e-04
Epoch 6/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 1112s 5s/step - accuracy: 0.2724 - loss: 4.9244 - val_accuracy: 0.5154 - val_loss: 2.0002 - learning_rate: 1.0000e-04
Epoch 7/50
226/226 ━━━━━━━━━━━━━━━━━━━━ 1101s 5s/ste

## 6 · Curvas de entrenamiento

`loss` es la pérdida de destilación y `val_loss` la CE del estudiante, así que
no son directamente comparables entre sí (sí lo son época a época).

In [ ]:
applyStyle()
import matplotlib.pyplot as plt

epocas = range(1, len(historia["loss"]) + 1)
figura, ejes = plt.subplots(1, 2, figsize=(11, 4.2))

ejes[0].plot(epocas, historia["accuracy"], color=SERIE_1, label="Entrenamiento")
ejes[0].plot(epocas, historia["val_accuracy"], color=SERIE_2, label="Validación")
ejes[0].set(title=f"{RUN_NAME} · Exactitud", xlabel="Época", ylim=(0, 1))
ejes[0].legend(loc="lower right")

ejes[1].plot(epocas, historia["loss"], color=SERIE_1, label="Destilación (train)")
ejes[1].plot(epocas, historia["val_loss"], color=SERIE_2, label="CE (validación)")
ejes[1].set(title=f"{RUN_NAME} · Pérdida", xlabel="Época")
ejes[1].legend(loc="upper right")

figura.tight_layout()
figura.savefig(IMGS / "training_history.png", bbox_inches="tight")
plt.show()

## 7 · Evaluación

In [ ]:
metricas = {
    "test": evalStats.evaluar(estudiante, dsTest, clases),
    "validate": evalStats.evaluar(estudiante, dsVal, clases),
}

pd.DataFrame([
    {k: round(v, 4) if isinstance(v, float) else v
     for k, v in m.items()
     if k in ("num_samples", "accuracy", "top3_accuracy",
              "macro_f1", "weighted_f1", "macro_precision", "macro_recall")}
    for m in metricas.values()
], index=list(metricas.keys()))

## 8 · ¿Sirvió la destilación?

Contra el mismo estudiante sin docente (su run de `01_student`) y contra el docente, que marca el techo.

In [ ]:
# el control debe compartir arquitectura Y régimen: si no, se compararía
# contra el estudiante congelado y el efecto del fine-tuning se atribuiría a la KD
controlRun = next(
    (r for r in registro.listRuns()
     if r["architecture"] == ARQUITECTURA
     and not r.get("teacher_run_id")
     and r.get("finetuning", 0.0) == FRACCION_DESCONGELADA),
    None,
)
if controlRun is None:
    raise ValueError(
        f"Falta el control: {ARQUITECTURA} sin destilar con "
        f"FRACCION_DESCONGELADA={FRACCION_DESCONGELADA}. Córrelo en 01_student/01.")

filas = [{
    "modelo": "docente",
    "arquitectura": ARQUITECTURA_DOCENTE,
    "test_accuracy": docenteInfo["test_accuracy"],
    "test_macro_f1": docenteInfo["test_macro_f1"],
    "parametros": docente.count_params(),
}]
if controlRun:
    filas.append({
        "modelo": "control (sin KD)",
        "arquitectura": ARQUITECTURA,
        "test_accuracy": controlRun["test_accuracy"],
        "test_macro_f1": controlRun["test_macro_f1"],
        "parametros": estudiante.count_params(),
    })
else:
    print("Sin run de control registrado — corre antes 01_student")
filas.append({
    "modelo": "estudiante (KD)",
    "arquitectura": ARQUITECTURA,
    "test_accuracy": metricas["test"]["accuracy"],
    "test_macro_f1": metricas["test"]["macro_f1"],
    "parametros": estudiante.count_params(),
})

comparacion = pd.DataFrame(filas)
if controlRun:
    delta = metricas["test"]["macro_f1"] - controlRun["test_macro_f1"]
    brecha = docenteInfo["test_macro_f1"] - controlRun["test_macro_f1"]
    print(f"Delta macro-F1 por destilar: {delta:+.4f}")
    if brecha > 0:
        print(f"Recupera el {delta / brecha:.1%} de la brecha estudiante-docente")
comparacion

## 9 · Matriz de confusión

In [ ]:
_ = plotConfusionMatrix(
    metricas["test"],
    f"Matriz de confusión · prueba · {RUN_NAME}",
)
plt.savefig(IMGS / "confusion_test.png", bbox_inches="tight")
plt.show()

## 10 · Confusiones principales

In [ ]:
confusiones = evalStats.confusionesPrincipales(metricas["test"])
pd.DataFrame(confusiones).head(10)

## 11 · Versionado

In [ ]:
estudiante.save(RUN_DIR / "model.keras")
registro.writeJson(RUN_DIR / "history.json", historia)
registro.writeJson(RUN_DIR / "metrics.json", {
    "run_id": RUN_ID,
    "class_names": clases,
    "splits": metricas,
    "top_confusions": {
        s: evalStats.confusionesPrincipales(m) for s, m in metricas.items()
    },
})

configuracion = {
    "run_id": RUN_ID,
    "name": RUN_NAME,
    "architecture": ARQUITECTURA,
    "role": ROL,
    "notes": NOTAS,
    "num_classes": len(clases),
    "distillation": {
        "tipo": "respuesta",
        "teacher_run_id": docenteInfo["run_id"],
        "teacher_architecture": ARQUITECTURA_DOCENTE,
        "temperatura": TEMPERATURA,
        "finetuning": FRACCION_DESCONGELADA,
        "alfa": ALFA,
    },
    "hyperparameters": {
        "img_size": list(TAMANO),
        "batch_size": LOTE,
        "epochs_budget": EPOCAS,
        "epochs_ran": len(historia["loss"]),
        "learning_rate": TASA_APRENDIZAJE,
        "early_stopping_patience": PACIENCIA,
        "class_weights": "balanced",
        "seed": SEMILLA,
        "backbone": f"{ARQUITECTURA}/imagenet (congelado)",
    },
    "environment": {
        "python": platform.python_version(),
        "tensorflow": tf.__version__,
        "platform": platform.platform(),
    },
    "started_at": inicio.isoformat(),
    "finished_at": fin.isoformat(),
    "training_minutes": round((fin - inicio).total_seconds() / 60, 1),
}
registro.writeJson(RUN_DIR / "run.json", configuracion)

registro.registerRun({
    "run_id": RUN_ID,
    "name": RUN_NAME,
    "architecture": ARQUITECTURA,
    "role": ROL,
    "teacher_run_id": docenteInfo["run_id"],
    "distillation": "respuesta",
    "finetuning": FRACCION_DESCONGELADA,
    "num_classes": len(clases),
    "epochs_ran": len(historia["loss"]),
    "training_minutes": configuracion["training_minutes"],
    "test_accuracy": metricas["test"]["accuracy"],
    "test_macro_f1": metricas["test"]["macro_f1"],
    "validate_accuracy": metricas["validate"]["accuracy"],
    "validate_macro_f1": metricas["validate"]["macro_f1"],
    "notes": NOTAS,
    "created_at": inicio.isoformat(),
})

if PROMOVER:
    registro.promoteRun(RUN_ID)

print(f"Run guardado en {RUN_DIR}")

## Resumen

In [ ]:
pd.Series({
    "run_id": RUN_ID,
    "estudiante": ARQUITECTURA,
    "docente": ARQUITECTURA_DOCENTE,
    "temperatura": TEMPERATURA,
    "alfa": ALFA,
    "epocas": len(historia["loss"]),
    "minutos": configuracion["training_minutes"],
    "test_accuracy": round(metricas["test"]["accuracy"], 4),
    "test_top3": round(metricas["test"]["top3_accuracy"], 4),
    "test_macro_f1": round(metricas["test"]["macro_f1"], 4),
    "validate_accuracy": round(metricas["validate"]["accuracy"], 4),
    "validate_macro_f1": round(metricas["validate"]["macro_f1"], 4),
}).to_frame("valor")